# Практика 29 · T5 і BART: усе як текст у текст> ⏱ **Зошит навчає чотири моделі.** Заміряно: близько трьох хвилин на 4 ядрах> без відеокарти. Якщо машина зайнята, стінний час буде більший — числа часу> нижче з процесорного годинника, вони від завантаження машини не залежать.Теорія — у [lecture.html](lecture.html). Тут ми **розвʼязуємо задачу**: беремоукраїнські переклади, що лежать у системі, і переписуємо задачу «чи цеповідомлення про помилку» у вигляді «дай текст — отримай текст».Що зробимо:1. зберемо корпус із файлів локалізації й побудуємо словник;2. перепишемо задачу класифікації в текстову форму;3. складемо три архітектури — енкодер, декодер, енкодер-декодер — і **зрівняємо   їх за кількістю ваг**;4. напишемо перехресну увагу руками й звіримо з бібліотечною;5. навчимо енкодер-декодер відновлювати вирізані проміжки — це передтренування   без жодної мітки;6. подивимось, як він **породжує** відповідь, крок за кроком;7. донавчимо його на мітках і порівняємо з енкодером, який має голову.Мережа не потрібна: усі дані вже лежать на машині, а моделі ми навчаємо самі.

In [ ]:
# Потоки фіксуємо ДО імпорту numpy: інакше очікування потоків OpenMP
# рахується як робота, і процесорний час роздувається в десятки разів.
import os
for v in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS'):
    os.environ[v] = '1'

import sys, time, math, random, glob, gettext, re, collections, gc
import numpy as np
import torch
import torch.nn as nn
torch.set_num_threads(1)

T_START = time.process_time()
print('Python  ', sys.version.split()[0])
print('torch   ', torch.__version__)
print('numpy   ', np.__version__)
print('ядер    ', os.cpu_count(), '(рахуємо в один потік)')

## 1 · Корпус: українські переклади, що вже лежать у системіКожна встановлена програма приносить файл `.mo` з перекладами. У ньому пари«англійський оригінал — український переклад». Це справжня українська мова,вона нічим не згенерована, і вона вже на диску — качати нічого не треба.Мітку «повідомлення про помилку» ставимо регулярним виразом по **англійському**оригіналу. Модель бачитиме тільки український бік, тож підглянути у відповідьвона не зможе.

In [ ]:
TOKEN = re.compile(r"[а-яїієґ]+(?:['ʼ’][а-яїієґ]+)*")
ERR = re.compile(r'\b(error|fail|failed|cannot|unable|invalid|denied|corrupt)\b', re.I)

def load_pairs():
    """Читаємо всі каталоги перекладів, які знайдемо на цій машині."""
    out = []
    for path in sorted(glob.glob('/usr/share/locale/uk/LC_MESSAGES/*.mo')):
        try:
            with open(path, 'rb') as f:
                catalog = gettext.GNUTranslations(f)
        except Exception:
            continue
        for src, dst in catalog._catalog.items():
            if isinstance(src, str) and isinstance(dst, str) \
               and len(dst) > 30 and 'Project-Id' not in dst:
                out.append((src, dst))
    return out

pairs = load_pairs()
rows = []
for src, dst in pairs:
    words = TOKEN.findall(dst.lower())
    if 3 <= len(words) <= 30:                  # надто короткі й надто довгі відкидаємо
        rows.append((words, 1 if ERR.search(src) else 0))

# Корпус лежить у порядку програм, тож перемішуємо ПЕРЕД поділом:
# інакше «перші 80 %» — це не менше даних, а вужчий набір програм.
random.Random(0).shuffle(rows)
print('прикладів        ', len(rows))
print('частка «помилка» ', round(sum(y for _, y in rows) / len(rows), 4))
print('медіана довжини  ', int(np.median([len(w) for w, _ in rows])), 'слів')

### СловникСлова, що трапились менш ніж пʼять разів, зливаємо в один токен `<unk>` —інакше словник роздувається хвостом із одноразових слів.Крім звичайних слів, у словнику будуть **службові токени**:- `<pad>` — заповнювач, щоб вирівняти речення в батчі;- `<bos>` — початок послідовності;- `<eos>` — кінець;- `<unk>` — незнайоме слово;- `<s0>`…`<s5>` — **сентинели**: «тут був вирізаний шматок».І ще дві дрібниці, які роблять усю тему можливою: слова-відповіді `помилка`і `звичайне` та слово-назва задачі `класифікуй` кладемо в **той самий** словник.Відповідь моделі — таке саме слово, як усі інші.

In [ ]:
PAD, BOS, EOS, UNK, SEP = 0, 1, 2, 3, 4
SENT0, NSENT = 5, 6                    # <s0>..<s5>
SPEC = ['<pad>', '<bos>', '<eos>', '<unk>', '<sep>'] + ['<s%d>' % i for i in range(NSENT)]

n = len(rows)
cut_train, cut_dev = int(0.8 * n), int(0.9 * n)
train_rows, dev_rows, test_rows = rows[:cut_train], rows[cut_train:cut_dev], rows[cut_dev:]

counts = collections.Counter(w for words, _ in train_rows for w in words)
vocab_words = [w for w, c in counts.most_common() if c >= 5]
for forced in ('помилка', 'звичайне', 'класифікуй'):
    if forced not in vocab_words:
        vocab_words.append(forced)

itos = SPEC + vocab_words
stoi = {w: i for i, w in enumerate(itos)}
V = len(itos)
MAXLEN = 24

ANS_ERR, ANS_OK = stoi['помилка'], stoi['звичайне']
PRE_CLS = stoi['класифікуй']

print('словник          ', V)
print('поділ 80/10/10   ', len(train_rows), len(dev_rows), len(test_rows))
print('id слів-відповідей', ANS_ERR, ANS_OK, '— звичайні позиції словника, не окремий простір')

## 2 · Задача теми «Класифікація текстів», переписана словамиРаніше ця задача виглядала так: на вхід речення, на вихід число 0 або 1.Тепер вона виглядає так: на вхід рядок, на вихід рядок. Подивимось на трисправжні приклади з корпусу.

In [ ]:
def to_ids(words, limit=MAXLEN):
    return [stoi.get(w, UNK) for w in words[:limit]]

def show(words, label):
    text = ' '.join(words[:12])
    print('вхід : класифікуй :', text)
    print('вихід:', 'помилка' if label else 'звичайне')
    print()

for words, label in train_rows[:3]:
    show(words, label)

print('Те саме числами — так це бачить модель:')
w, y = train_rows[0]
print('  джерело', [BOS, PRE_CLS] + to_ids(w)[:6], '...')
print('  ціль   ', [ANS_ERR if y else ANS_OK])

## 3 · Три архітектури поручТепер складемо три моделі. Різниця між ними — **лише в тому, кому дозволенодивитись на кого** і що виходить назовні:| | вхід читає | вихід ||---|---|---|| енкодер | в обидва боки | вектори + маленька голова на 2 класи || декодер | лише ліворуч | розподіл по всьому словнику || енкодер-декодер | в обидва боки | розподіл по всьому словнику |Порівнювати їх «як є» не можна: блок декодера дорожчий за блок енкодера наперехресну увагу. Тому одностосовим моделям ми дамо **вдвічі більше блоків**і трохи ширший внутрішній шар — щоб кількість ваг у тілі збіглася.

In [ ]:
D_MODEL, HEADS = 128, 4

def enc_layer(ff):
    return nn.TransformerEncoderLayer(D_MODEL, HEADS, dim_feedforward=ff,
                                      batch_first=True, dropout=0.0, norm_first=True)

class EncoderOnly(nn.Module):
    """Читає в обидва боки, тексту не пише. Відповідь дає голова на 2 класи."""
    def __init__(self, layers=4, ff=641):
        super().__init__()
        self.emb = nn.Embedding(V, D_MODEL, padding_idx=PAD)
        self.pos = nn.Embedding(2 * MAXLEN + 8, D_MODEL)
        self.body = nn.TransformerEncoder(enc_layer(ff), layers, enable_nested_tensor=False)
        self.head = nn.Linear(D_MODEL, 2)
    def forward(self, x):
        h = self.emb(x) + self.pos(torch.arange(x.size(1)))
        h = self.body(h, src_key_padding_mask=(x == PAD))
        return self.head(h[:, 0])            # позиція 0 грає роль [CLS]

class DecoderOnly(nn.Module):
    """Бачить лише ліве. Відповідь — слово, обране з усього словника."""
    def __init__(self, layers=4, ff=641):
        super().__init__()
        self.emb = nn.Embedding(V, D_MODEL, padding_idx=PAD)
        self.pos = nn.Embedding(2 * MAXLEN + 8, D_MODEL)
        self.body = nn.TransformerEncoder(enc_layer(ff), layers, enable_nested_tensor=False)
        self.lm = nn.Linear(D_MODEL, V)
    def hidden(self, x):
        T = x.size(1)
        h = self.emb(x) + self.pos(torch.arange(T))
        causal = torch.triu(torch.ones(T, T, dtype=torch.bool), 1)   # заборона дивитись праворуч
        return self.body(h, mask=causal, src_key_padding_mask=(x == PAD), is_causal=False)
    def at(self, x, position):
        h = self.hidden(x)
        return self.lm(h[torch.arange(x.size(0)), position])

class EncoderDecoder(nn.Module):
    """Енкодер читає джерело в обидва боки, декодер пише ціль зліва направо
       й на кожному кроці дивиться в памʼять енкодера."""
    def __init__(self, enc_layers=2, dec_layers=2, ff=512):
        super().__init__()
        self.emb = nn.Embedding(V, D_MODEL, padding_idx=PAD)
        self.pos = nn.Embedding(2 * MAXLEN + 8, D_MODEL)
        self.enc = nn.TransformerEncoder(enc_layer(ff), enc_layers, enable_nested_tensor=False)
        dl = nn.TransformerDecoderLayer(D_MODEL, HEADS, dim_feedforward=ff,
                                        batch_first=True, dropout=0.0, norm_first=True)
        self.dec = nn.TransformerDecoder(dl, dec_layers)
        self.lm = nn.Linear(D_MODEL, V)
    def encode(self, x):
        h = self.emb(x) + self.pos(torch.arange(x.size(1)))
        return self.enc(h, src_key_padding_mask=(x == PAD))
    def decode(self, memory, src_pad, y):
        T = y.size(1)
        g = self.emb(y) + self.pos(torch.arange(T))
        causal = torch.triu(torch.ones(T, T, dtype=torch.bool), 1)
        g = self.dec(g, memory, tgt_mask=causal, tgt_key_padding_mask=(y == PAD),
                     memory_key_padding_mask=src_pad)
        return self.lm(g)
    def forward(self, x, y):
        return self.decode(self.encode(x), (x == PAD), y)

def body_params(model):
    """Ваги власне архітектури: без таблиць ембедингів і без вихідного шару."""
    skip = ('emb', 'pos', 'lm', 'head')
    return sum(p.numel() for name, p in model.named_parameters()
               if not name.startswith(skip))

models = {'енкодер': EncoderOnly(), 'декодер': DecoderOnly(), 'енкодер-декодер': EncoderDecoder()}
for name, m in models.items():
    total = sum(p.numel() for p in m.parameters())
    print(f'{name:18s} тіло {body_params(m):>9,} · усього {total:>10,}'.replace(',', ' '))

Тіла збіглися до кількох ваг — тепер порівняння вимірює будову, а не розмір.А от **усього** ваг у них різна кількість, і різниця величезна. Причина одна:енкодерові досить голови на два числа, а моделі, що відповідає словом, потрібнапроєкція на **весь словник**. Порахуймо це прямо.

In [ ]:
enc, ed = models['енкодер'], models['енкодер-декодер']
p_head = sum(p.numel() for p in enc.head.parameters())
p_lm = sum(p.numel() for p in ed.lm.parameters())
print('голова на 2 класи          ', f'{p_head:,}'.replace(',', ' '), 'ваг')
print('проєкція на весь словник   ', f'{p_lm:,}'.replace(',', ' '), 'ваг')
print('різниця у', round(p_lm / p_head), 'разів — ось ціна того, щоб відповідати словом')

## 4 · Перехресна увага рукамиУся новизна енкодер-декодера — один рядок: **запити беруться з декодера,а ключі й значення — з енкодера**. Напишімо це формулою й звіримо з бібліотечним`nn.MultiheadAttention`. Якщо збігається — значить усередині бібліотеки самете, що ми щойно написали, і ніякої магії там немає.

In [ ]:
torch.manual_seed(0)
d, n_src, n_tgt = 16, 5, 3
q_from_decoder = torch.randn(1, n_tgt, d)       # що шукає той, хто пише
kv_from_encoder = torch.randn(1, n_src, d)      # що пропонує той, хто прочитав

attn = nn.MultiheadAttention(d, num_heads=1, bias=False, batch_first=True)
with torch.no_grad():
    Wq, Wk, Wv = attn.in_proj_weight.chunk(3)   # бібліотека тримає три матриці разом
    Wo = attn.out_proj.weight
    Q = q_from_decoder @ Wq.T
    K = kv_from_encoder @ Wk.T
    V_ = kv_from_encoder @ Wv.T
    scores = Q @ K.transpose(1, 2) / math.sqrt(d)   # схожість кожного запиту з кожним ключем
    weights = torch.softmax(scores, dim=-1)         # частки, що додаються в одиницю
    ours = (weights @ V_) @ Wo.T
    theirs, _ = attn(q_from_decoder, kv_from_encoder, kv_from_encoder, need_weights=False)

gap = float((ours - theirs).abs().max())
print('таблиця ваг має форму', tuple(weights.shape[1:]), '— прямокутну, а не квадратну')
print('перший рядок ваг     ', np.round(weights[0, 0].numpy(), 4), ' сума', round(float(weights[0, 0].sum()), 6))
print('max |різниця|        ', f'{gap:.3e}')
print('машинний епсилон     ', f'{torch.finfo(torch.float32).eps:.3e}')
assert gap < torch.finfo(torch.float32).eps, 'розрахунок розійшовся!'
print('✅ збігається з точністю до однієї одиниці останнього розряду')

Форма таблиці ваг — `(3, 5)`: три слова, які пишемо, проти пʼятьох, якіпрочитали. У самоуваги вона була б квадратною. Саме ця прямокутність і є«міст між стосами».## 5 · Псування тексту: мітки задармаТепер передтренування. Мітки нам не потрібні: беремо речення, вирізаємо з ньогокілька шматків і просимо модель написати те, що вирізали. Це рецепт T5: у вхідзамість шматка ставиться **один** сентинел, а ціль складається лише з вирізаного.

In [ ]:
def corrupt_spans(word_ids, rng, rate=0.15, mean_len=3.0, max_spans=NSENT):
    """Вирізає з речення кілька шматків. Повертає (зіпсоване, що треба відновити)."""
    n = len(word_ids)
    need = max(1, int(round(rate * n)))          # скільки слів усього ховаємо
    taken = [False] * n
    spans, guard = [], 0
    while sum(len(s) for s in spans) < need and len(spans) < max_spans and guard < 40:
        guard += 1
        length = min(max(1, int(rng.expovariate(1.0 / mean_len)) + 1), need)
        if length > n:
            continue
        start = rng.randrange(0, n - length + 1)
        # шматки не мають ані перетинатись, ані торкатись — інакше вони зіллються в один
        if any(taken[max(0, start - 1):start + length + 1]):
            continue
        for k in range(start, start + length):
            taken[k] = True
        spans.append(list(range(start, start + length)))
    if not spans:
        return None
    spans.sort(key=lambda s: s[0])
    source, target, i, si = [], [], 0, 0
    while i < n:
        if si < len(spans) and i == spans[si][0]:
            source.append(SENT0 + si)                 # у вході — один токен на цілий шматок
            target.append(SENT0 + si)                 # у цілі — той самий сентинел і вміст
            target.extend(word_ids[k] for k in spans[si])
            i += len(spans[si]); si += 1
        else:
            source.append(word_ids[i]); i += 1
    target.append(EOS)
    return source, target

rng = random.Random(7)
example = to_ids(next(w for w, _ in train_rows if len(w) >= 10))
src, tgt = corrupt_spans(example, rng)
print('оригінал :', ' '.join(itos[t] for t in example))
print('вхід     :', ' '.join(itos[t] for t in src))
print('ціль     :', ' '.join(itos[t] for t in tgt))
print()
print('токенів: оригінал', len(example), '· вхід', len(src), '· ціль', len(tgt))

### Скільки нат коштує вгадати закрите слово навманняЩоб знати, чи модель узагалі чогось навчилась, потрібен рубіж. Найпростіший —**уніграмний**: модель, яка не дивиться на контекст і називає слова за їхньоючастотою. Порахуємо його на тих самих закритих словах.

In [ ]:
def pad_batch(seqs):
    L = max(len(s) for s in seqs)
    return torch.tensor([s + [PAD] * (L - len(s)) for s in seqs], dtype=torch.long)

train_ids = [to_ids(w) for w, _ in train_rows]
dev_ids = [to_ids(w) for w, _ in dev_rows]

freq = collections.Counter(t for s in train_ids for t in s)
total = sum(freq.values())
unigram = torch.full((V,), 1e-12)
for k, c in freq.items():
    unigram[k] = c / total

rng = random.Random(99)
dev_pairs = [p for p in (corrupt_spans(s, rng) for s in dev_ids[:1500]) if p]

def hidden_tokens(pairs):
    """Лише закриті СЛОВА: сентинели й кінець рядка передбачити легко,
       і вони розмили б число."""
    return [t for _, tg in pairs for t in tg
            if t != EOS and not (SENT0 <= t < SENT0 + NSENT)]

hid = hidden_tokens(dev_pairs)
UNIGRAM_BOUND = float(-torch.log(unigram[torch.tensor(hid)]).mean())
print('закритих слів у відкладеній   ', len(hid))
print('уніграмний рубіж, нат         ', round(UNIGRAM_BOUND, 4))
print('вгадування навмання, нат      ', round(math.log(V), 4), '(це ln від розміру словника)')

## 6 · Передтренування: відновлюємо вирізанеНавчаємо енкодер-декодер відновлювати проміжки. Жодної мітки тут немає —відповіді беруться з самого тексту.Дивитись будемо не на загальну втрату, а на втрату **саме на закритих словах**:тільки її можна чесно порівняти з уніграмним рубежем.

In [ ]:
@torch.no_grad()
def hidden_loss(model, pairs, batch=128):
    """Середня втрата на закритих словах, у натах."""
    model.eval()
    s, n = 0.0, 0
    for i in range(0, len(pairs), batch):
        b = pairs[i:i + batch]
        x = pad_batch([[BOS] + a for a, _ in b])
        y = pad_batch([[BOS] + t for _, t in b])
        logp = torch.log_softmax(model(x, y[:, :-1]), -1)
        gold = y[:, 1:]
        picked = logp.gather(-1, gold.unsqueeze(-1)).squeeze(-1)
        keep = (gold != PAD) & (gold != EOS) & ~((gold >= SENT0) & (gold < SENT0 + NSENT))
        s += float((-picked * keep).sum()); n += int(keep.sum())
    model.train()
    return s / n

def pretrain(steps=400, batch=64, lr=0.001, seed=0, report=(100, 200, 400)):
    torch.manual_seed(seed)
    rg = random.Random(seed)
    model = EncoderDecoder()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    lossf = nn.CrossEntropyLoss(ignore_index=PAD)
    t0 = time.process_time()
    for step in range(1, steps + 1):
        batch_pairs = []
        while len(batch_pairs) < batch:
            p = corrupt_spans(rg.choice(train_ids), rg)
            if p:
                batch_pairs.append(p)
        x = pad_batch([[BOS] + a for a, _ in batch_pairs])
        y = pad_batch([[BOS] + t for _, t in batch_pairs])
        loss = lossf(model(x, y[:, :-1]).reshape(-1, V), y[:, 1:].reshape(-1))
        opt.zero_grad(); loss.backward(); opt.step()
        if step in report:
            hl = hidden_loss(model, dev_pairs)
            mark = 'нижче рубежу' if hl < UNIGRAM_BOUND else 'ВИЩЕ рубежу'
            print(f'  крок {step:4d} · втрата на закритих словах {hl:.4f} нат · {mark}')
    return model, time.process_time() - t0

print('уніграмний рубіж:', round(UNIGRAM_BOUND, 4), 'нат\n')
pretrained, sec_pre = pretrain()
print(f'\nпередтренування: {sec_pre:.1f} с процесорного часу')

Поки втрата **вища** за уніграмний рубіж, моделі фактично ще немає: простатаблиця частот вгадує краще за неї. Порівнювати з такою моделлю немає сенсу —це вимірювання шуму. Нижче рубежу вона нарешті починає користуватись контекстом.## 7 · Як воно породжує: крок за крокомТепер найважливіше про декодер: **він пише по одному токену за раз**. Памʼятьенкодера рахується один раз, а декодер запускається стільки разів, скількитокенів у відповіді. Подивимось на це буквально.

In [ ]:
@torch.no_grad()
def restore(model, source_ids, max_new=8, verbose=True):
    """Відновлює вирізане, дописуючи по одному токену."""
    model.eval()
    x = torch.tensor([[BOS] + source_ids])
    memory = model.encode(x)                  # рахується ОДИН раз
    y = torch.tensor([[BOS]])
    runs = 0
    for _ in range(max_new):
        logits = model.decode(memory, (x == PAD), y)
        nxt = int(logits[0, -1].argmax())
        runs += 1
        y = torch.cat([y, torch.tensor([[nxt]])], 1)
        if verbose:
            print(f'  прогін {runs}: дописано «{itos[nxt]}»')
        if nxt == EOS:
            break
    model.train()
    return [int(t) for t in y[0, 1:]], runs

src, tgt = dev_pairs[3]
print('вхід    :', ' '.join(itos[t] for t in src))
print('еталон  :', ' '.join(itos[t] for t in tgt))
print('породження:')
out, runs = restore(pretrained, src)
print('\nмодель написала:', ' '.join(itos[t] for t in out))
print('енкодер запущено 1 раз, декодер —', runs, 'разів')

Модель на такому бюджеті ще не вгадує слова точно — і це нормально: вонабачила чотириста батчів, а не мільярд. Важливе тут інше: **механізм**. Коженрядок «прогін N» — це окремий запуск декодера, і паралельно їх зробитинеможливо, бо наступний токен залежить від попереднього.Порівняймо ціну породження з ціною одного прогону, у якому вся відповідь відоманаперед (так відбувається під час навчання — це називають **подаванням еталона**).

In [ ]:
batch_src = [s for s, _ in dev_pairs[:64]]
x = pad_batch([[BOS] + s for s in batch_src])

with torch.no_grad():
    t0 = time.process_time()
    for _ in range(3):
        memory = pretrained.encode(x)
        y = torch.full((len(batch_src), 1), BOS, dtype=torch.long)
        for _k in range(8):                       # вісім послідовних кроків
            logits = pretrained.decode(memory, (x == PAD), y)
            y = torch.cat([y, logits[:, -1:].argmax(-1)], 1)
    sequential = (time.process_time() - t0) / 3

    t0 = time.process_time()
    for _ in range(3):
        pretrained(x, torch.full((len(batch_src), 8), BOS, dtype=torch.long))
    with_teacher = (time.process_time() - t0) / 3

print(f'вісім токенів послідовно      {sequential * 1000:7.1f} мс на батч із 64')
print(f'ті самі вісім одним прогоном  {with_teacher * 1000:7.1f} мс')
print(f'послідовне дорожче у {sequential / with_teacher:.1f} раза')

## 8 · Донавчання: та сама модель відповідає словомТепер даємо мітки. Відповідь — слово «помилка» або «звичайне», і модель обираєйого тим самим механізмом, яким щойно відновлювала вирізане. Ніякої новоїголови не додається.Порівнюємо три речі на **однаковому** бюджеті й однакових мітках:- передтренований енкодер-декодер;- такий самий, але з випадкових ваг;- енкодер із головою на два класи — той самий підхід, що в попередніх темах.

In [ ]:
N_LABELS = 4000
# Вибірка міток ВИПАДКОВА, а не «перші N»: корпус лежить у порядку програм,
# тож префікс дав би вужчий домен, а не менше даних.
labelled = random.Random(1).sample(train_rows, N_LABELS)
test_sample = test_rows

def batch_encdec(items):
    x = pad_batch([[BOS, PRE_CLS] + to_ids(w) for w, _ in items])
    y_in = torch.full((len(items), 1), BOS, dtype=torch.long)
    ans = torch.tensor([ANS_ERR if y else ANS_OK for _, y in items])
    return x, y_in, ans

def batch_enconly(items):
    x = pad_batch([[BOS] + to_ids(w) for w, _ in items])
    return x, torch.tensor([y for _, y in items])

def f1_positive(pred, gold):
    tp = sum(1 for p, g in zip(pred, gold) if p == 1 and g == 1)
    fp = sum(1 for p, g in zip(pred, gold) if p == 1 and g == 0)
    fn = sum(1 for p, g in zip(pred, gold) if p == 0 and g == 1)
    return 0.0 if tp == 0 else 2 * tp / (2 * tp + fp + fn)

def finetune(model, kind, steps=300, batch=32, lr=0.0003, seed=0):
    torch.manual_seed(seed + 100)
    rg = random.Random(seed + 100)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    lossf = nn.CrossEntropyLoss()
    t0 = time.process_time()
    for _ in range(steps):
        b = rg.sample(labelled, batch)
        if kind == 'ed':
            x, y_in, ans = batch_encdec(b)
            loss = lossf(model(x, y_in)[:, 0], ans)
        else:
            x, y = batch_enconly(b)
            loss = lossf(model(x), y)
        opt.zero_grad(); loss.backward(); opt.step()
    return model, time.process_time() - t0

@torch.no_grad()
def evaluate(model, kind, data, batch=256):
    model.eval()
    pred, gold, outside = [], [], 0
    two = torch.tensor([ANS_OK, ANS_ERR])
    for i in range(0, len(data), batch):
        b = data[i:i + batch]
        if kind == 'ed':
            x, y_in, _ = batch_encdec(b)
            out = model(x, y_in)[:, 0]
            pick = out[:, [ANS_OK, ANS_ERR]].argmax(1)
            outside += int((out.argmax(1) != two[pick]).sum())
            pred += pick.tolist()
        else:
            x, _ = batch_enconly(b)
            pred += model(x).argmax(1).tolist()
        gold += [y for _, y in b]
    model.train()
    return f1_positive(pred, gold), outside / len(data)

results = {}
model, sec = finetune(pretrained, 'ed')
results['енкодер-декодер, передтренований'] = evaluate(model, 'ed', test_sample) + (sec,)
del model; gc.collect()

torch.manual_seed(0)
model, sec = finetune(EncoderDecoder(), 'ed')
results['енкодер-декодер, з нуля'] = evaluate(model, 'ed', test_sample) + (sec,)
del model; gc.collect()

torch.manual_seed(0)
model, sec = finetune(EncoderOnly(), 'enc')
results['енкодер + голова'] = evaluate(model, 'enc', test_sample) + (sec,)
del model; gc.collect()

print(f'{N_LABELS} міток, 300 кроків, крок навчання 0.0003, одне зерно\n')
for name, (f1, outside, sec) in results.items():
    print(f'{name:36s} F1 {f1:.4f} · {sec:5.1f} с')

### Скільки разів відповідь вийшла за межі двох дозволених слівМодель вибирає з **усього** словника, тож теоретично могла б відповісти будь-якимсловом. Перевіримо, чи це трапляється.

In [ ]:
for name, (f1, outside, sec) in results.items():
    if 'енкодер-декодер' in name:
        print(f'{name:36s} поза двома словами: {outside:.4%}')
print()
print('Разом зошит витратив', round(time.process_time() - T_START, 1), 'с процесорного часу.')

⚠️ **Одне зерно й один крок навчання — це не результат, і зараз добре видночому.** Передтренована модель вийшла тут **гіршою** за ту, що починала звипадкових ваг, — хоча передтренування мало б допомагати. Причин дві, і обидвіметодичні: по-перше, кожну модель запущено рівно один раз, а розкид між зернамина такому обсязі міток більший за цю різницю; по-друге, крок навчання в усіхтрьох однаковий, тоді як передтренованій моделі зазвичай потрібен менший.Різниця, менша за розкид між зернами, різницею не є. У лекції те саме порівняннязроблене на трьох зернах і з кроком, дібраним **кожній гілці окремо** навідкладеній частині, — і там воно виглядає інакше. Тут ми дивимось на**механізм**, а не на переможця.---## Завдання### 🟢 Рівень 1 — БазаЗаміни пару слів-відповідей: замість «помилка» / «звичайне» візьми «так» / «ні»(додай їх у словник так само, як ми додали попередні) і перезапусти донавчанняенкодер-декодера.**Зроблено, якщо:** надрукував F1 для обох пар слів і пояснив словами, чомурізниця (або її відсутність) саме така, спираючись на частоту цих слів у корпусі.### 🟡 Рівень 2 — ПлюсПостав поруч два рецепти псування: наш `corrupt_spans` із середньою довжиноюшматка 3 і його ж із середньою довжиною 1 (тобто окремі слова). Навчи по моделіна кожному, однаковий бюджет, і зміряй `hidden_loss` кожної **на її власному**наборі.**Зроблено, якщо:** побудував таблицю з двома втратами й двома уніграмнимирубежами й сказав, яка форма дірки складніша та на скільки нат.### 🔴 Рівень 3 — ВикликНаш `restore` бере на кожному кроці найімовірніший токен — це жадібнедекодування. Реалізуй **промінь** (beam search) шириною 3: тримай тринайкращі часткові відповіді, на кожному кроці продовжуй кожну й лишай тринайкращі за сумою логарифмів імовірностей.**Зроблено, якщо:** на двадцяти прикладах із відкладеної частини порівнявжадібне декодування з променем за сумарною логарифмічною імовірністювідповіді й назвав, скільки разів промінь знайшов кращу.## Підказки- Слово потрапляє у словник лише якщо трапилось пʼять разів; для рівня 1  подивись `counts['так']` і `counts['ні']` перед тим, як робити висновки.- Для рівня 2 памʼятай: рубіж треба рахувати **на своєму** наборі пар, бо  набір закритих слів у двох рецептах різний.- Для рівня 3 найпростіше тримати промінь списком пар `(сума_логімовірностей,  список_токенів)` і на кожному кроці робити один прогін декодера на всі три  гілки одразу, склавши їх у батч.